# 04. Hyperparameter Tuning for Top Models

This notebook tunes the strongest model families from the staged early-warning benchmark.

Tuned models:
- Random Forest
- XGBoost
- LightGBM
- CatBoost
- RBF SVM
- Neural Network / MLP

Selection rule: choose the best validation Macro-F1 within each stage and model family, using balanced accuracy as a secondary criterion and log-loss as a probability-quality check.

## 1. Imports, Paths, and Folders

In [ ]:
from pathlib import Path
import json
import pickle
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import ParameterGrid
from sklearn.preprocessing import LabelEncoder

from models.model_registry import MODEL_REGISTRY, build_model
from models.preprocessing import TabularOneHotScaler
from models.evaluation import classification_metrics

warnings.filterwarnings("ignore")

RESULTS_DIR = Path("results")
PREP_DIR = RESULTS_DIR / "2.Model Processing"
TUNE_DIR = RESULTS_DIR / "4.Hyperparameter Tuning"
TUNE_CSV_DIR = TUNE_DIR / "csv"
TUNE_PLOTS_DIR = TUNE_DIR / "plots"
TUNE_REPORTS_DIR = TUNE_DIR / "reports"
TUNE_MODELS_DIR = TUNE_DIR / "models"

for folder in [TUNE_CSV_DIR, TUNE_PLOTS_DIR, TUNE_REPORTS_DIR, TUNE_MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MASTER_DATA_PATH = PREP_DIR / "csv" / "modeling_master_dataset_with_splits.csv"
FEATURE_METADATA_PATH = PREP_DIR / "metadata" / "feature_sets_metadata.json"
TUNE_DIR

## 2. Load Data and Metadata

In [ ]:
df = pd.read_csv(MASTER_DATA_PATH)
metadata = json.loads(FEATURE_METADATA_PATH.read_text(encoding="utf-8"))

stage_feature_sets = metadata["stage_feature_sets"]
categorical_features_all = metadata["feature_types"]["categorical_features"]
numeric_features_all = metadata["feature_types"]["numeric_features"]
target_multiclass = metadata["targets"]["multiclass"]
split_column = metadata["split_column"]

label_encoder = LabelEncoder()
label_encoder.fit(df[target_multiclass])
class_labels = label_encoder.classes_.tolist()

display(df[[split_column, target_multiclass]].value_counts().sort_index().rename("n").reset_index())
display(pd.DataFrame([{"stage": k, "n_features": len(v)} for k, v in stage_feature_sets.items()]))

## 3. Preprocessing Helpers

In [ ]:
def get_stage_feature_types(stage_features):
    categorical = [feature for feature in stage_features if feature in categorical_features_all]
    numeric = [feature for feature in stage_features if feature in numeric_features_all]
    missing = sorted(set(stage_features) - set(categorical) - set(numeric))
    if missing:
        raise ValueError(f"Features missing type metadata: {missing}")
    return numeric, categorical


def preprocess_stage(stage_features):
    numeric, categorical = get_stage_feature_types(stage_features)
    train_df = df[df[split_column] == "train"].copy()
    validation_df = df[df[split_column] == "validation"].copy()
    test_df = df[df[split_column] == "test"].copy()

    preprocessor = TabularOneHotScaler(numeric, categorical)
    X_train = preprocessor.fit_transform(train_df[stage_features]).to_numpy(np.float32)
    X_validation = preprocessor.transform(validation_df[stage_features]).to_numpy(np.float32)
    X_test = preprocessor.transform(test_df[stage_features]).to_numpy(np.float32)

    y_train = label_encoder.transform(train_df[target_multiclass])
    y_validation = label_encoder.transform(validation_df[target_multiclass])
    y_test = label_encoder.transform(test_df[target_multiclass])

    y_train_labels = train_df[target_multiclass].to_numpy()
    y_validation_labels = validation_df[target_multiclass].to_numpy()
    y_test_labels = test_df[target_multiclass].to_numpy()

    return {
        "preprocessor": preprocessor,
        "X_train": X_train,
        "X_validation": X_validation,
        "X_test": X_test,
        "y_train": y_train,
        "y_validation": y_validation,
        "y_test": y_test,
        "y_train_labels": y_train_labels,
        "y_validation_labels": y_validation_labels,
        "y_test_labels": y_test_labels,
    }

## 4. Tuning Grids

In [ ]:
top_model_keys = ["random_forest", "xgboost", "lightgbm", "catboost", "rbf_svm", "mlp"]

tuning_grids = {
    "random_forest": {
        "n_estimators": [300, 500],
        "max_depth": [None, 12],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt"],
    },
    "xgboost": {
        "n_estimators": [250, 400],
        "learning_rate": [0.03, 0.05],
        "max_depth": [3, 4],
        "subsample": [0.9],
        "colsample_bytree": [0.9],
    },
    "lightgbm": {
        "n_estimators": [200, 400],
        "learning_rate": [0.03, 0.05],
        "num_leaves": [15, 31],
        "max_depth": [5, -1],
    },
    "catboost": {
        "iterations": [300, 500],
        "learning_rate": [0.03, 0.05],
        "depth": [4, 6],
        "l2_leaf_reg": [3],
    },
    "rbf_svm": {
        "C": [0.5, 1.0, 3.0],
        "gamma": ["scale", 0.03],
    },
    "mlp": {
        "hidden_layer_sizes": [(64,), (64, 32), (128, 64)],
        "activation": ["relu", "tanh"],
        "alpha": [0.0001, 0.001],
        "learning_rate_init": [0.001],
    },
}

grid_summary = pd.DataFrame([
    {"model_key": key, "model": MODEL_REGISTRY[key].display_name, "n_candidates": len(list(ParameterGrid(grid)))}
    for key, grid in tuning_grids.items()
])
grid_summary.to_csv(TUNE_CSV_DIR / "tuning_grid_summary.csv", index=False)
grid_summary

## 5. Candidate Evaluation Helpers

In [ ]:
def evaluate_model(model, prepared):
    outputs = {}
    for split_name, X, y_labels in [
        ("train", prepared["X_train"], prepared["y_train_labels"]),
        ("validation", prepared["X_validation"], prepared["y_validation_labels"]),
        ("test", prepared["X_test"], prepared["y_test_labels"]),
    ]:
        pred_encoded = np.asarray(model.predict(X)).reshape(-1).astype(int)
        pred_labels = label_encoder.inverse_transform(pred_encoded)
        proba = model.predict_proba(X)
        outputs[split_name] = classification_metrics(y_labels, pred_labels, y_proba=proba)
    return outputs


def fit_candidate(model_key, params, prepared):
    model = build_model(model_key, **params)
    start = time.time()
    model.fit(prepared["X_train"], prepared["y_train"])
    fit_seconds = time.time() - start
    return model, fit_seconds

## 6. Run Tuning

In [ ]:
candidate_rows = []
best_metric_rows = []
failures = []

for stage_name, stage_features in stage_feature_sets.items():
    print(f"\nStage: {stage_name}")
    prepared = preprocess_stage(stage_features)

    for model_key in top_model_keys:
        spec = MODEL_REGISTRY[model_key]
        print(f"  Tuning {spec.display_name}")
        best_score_tuple = None
        best_model = None
        best_params = None
        best_fit_seconds = None

        for candidate_id, params in enumerate(ParameterGrid(tuning_grids[model_key]), start=1):
            try:
                model, fit_seconds = fit_candidate(model_key, params, prepared)
                outputs = evaluate_model(model, prepared)
                validation_metrics = outputs["validation"]
                score_tuple = (
                    validation_metrics["macro_f1"],
                    validation_metrics["balanced_accuracy"],
                    -validation_metrics["log_loss"],
                )

                candidate_row = {
                    "stage": stage_name,
                    "model_key": model_key,
                    "model": spec.display_name,
                    "candidate_id": candidate_id,
                    "params": json.dumps(params),
                    "fit_seconds": fit_seconds,
                }
                candidate_row.update({f"validation_{k}": v for k, v in validation_metrics.items()})
                candidate_rows.append(candidate_row)

                if best_score_tuple is None or score_tuple > best_score_tuple:
                    best_score_tuple = score_tuple
                    best_model = model
                    best_params = params
                    best_fit_seconds = fit_seconds

            except Exception as exc:
                failures.append({
                    "stage": stage_name,
                    "model_key": model_key,
                    "model": spec.display_name,
                    "candidate_id": candidate_id,
                    "params": json.dumps(params),
                    "reason": repr(exc),
                })

        if best_model is not None:
            model_path = TUNE_MODELS_DIR / f"{stage_name}__{model_key}.pkl"
            with open(model_path, "wb") as f:
                pickle.dump(best_model, f)

            outputs = evaluate_model(best_model, prepared)
            for split_name, metrics in outputs.items():
                row = {
                    "stage": stage_name,
                    "model_key": model_key,
                    "model": spec.display_name,
                    "split": split_name,
                    "raw_features": len(stage_features),
                    "encoded_features": prepared["X_train"].shape[1],
                    "best_params": json.dumps(best_params),
                    "fit_seconds": best_fit_seconds,
                    "model_path": str(model_path),
                }
                row.update(metrics)
                best_metric_rows.append(row)

candidate_results = pd.DataFrame(candidate_rows)
tuned_results = pd.DataFrame(best_metric_rows)
failure_results = pd.DataFrame(failures)

candidate_results.to_csv(TUNE_CSV_DIR / "tuning_candidate_validation_results.csv", index=False)
tuned_results.to_csv(TUNE_CSV_DIR / "tuned_model_comparison.csv", index=False)
failure_results.to_csv(TUNE_CSV_DIR / "tuning_failures.csv", index=False)

display(tuned_results)
display(failure_results)

## 7. Best Tuned Model by Stage

In [ ]:
validation_tuned = tuned_results[tuned_results["split"] == "validation"].copy()
best_by_stage = (
    validation_tuned.sort_values(
        ["stage", "macro_f1", "balanced_accuracy", "log_loss"],
        ascending=[True, False, False, True],
    )
    .groupby("stage", as_index=False)
    .head(1)
)
best_by_stage.to_csv(TUNE_CSV_DIR / "best_tuned_validation_model_by_stage.csv", index=False)
best_by_stage

## 8. Overfitting Gap Table

In [ ]:
metric_wide = tuned_results.pivot_table(
    index=["stage", "model_key", "model"],
    columns="split",
    values=["macro_f1", "balanced_accuracy", "log_loss"],
    aggfunc="first",
)
metric_wide.columns = [f"{metric}_{split}" for metric, split in metric_wide.columns]
metric_wide = metric_wide.reset_index()
metric_wide["macro_f1_train_validation_gap"] = metric_wide["macro_f1_train"] - metric_wide["macro_f1_validation"]
metric_wide["macro_f1_validation_test_gap"] = metric_wide["macro_f1_validation"] - metric_wide["macro_f1_test"]
metric_wide["log_loss_validation_test_gap"] = metric_wide["log_loss_validation"] - metric_wide["log_loss_test"]
metric_wide.to_csv(TUNE_CSV_DIR / "tuned_overfitting_gap_table.csv", index=False)
metric_wide

## 9. Plots

In [ ]:
stage_order = list(stage_feature_sets.keys())
stage_labels = {
    "A_pre_enrollment": "A: Pre-enrollment",
    "B_first_semester": "B: + First semester",
    "C_second_semester": "C: + Second semester",
}

for metric, label in [("macro_f1", "Macro-F1"), ("balanced_accuracy", "Balanced Accuracy"), ("log_loss", "Log-Loss")]:
    for split in ["train", "validation", "test"]:
        sub = tuned_results[tuned_results["split"] == split].copy()
        plt.figure(figsize=(11, 6.5))
        for model, grp in sub.groupby("model"):
            grp = grp.set_index("stage").reindex(stage_order).reset_index()
            plt.plot([stage_labels[s] for s in grp["stage"]], grp[metric], marker="o", linewidth=2, label=model)
        plt.title(f"Tuned {label} Across Early-Warning Stages ({split.title()})")
        plt.xlabel("Information stage")
        plt.ylabel(label)
        plt.xticks(rotation=15, ha="right")
        plt.grid(True, alpha=0.25)
        plt.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
        plt.tight_layout()
        plt.savefig(TUNE_PLOTS_DIR / f"{split}_{metric}_by_stage_tuned.png", dpi=220)
        plt.close()

for split in ["validation", "test"]:
    for metric, label in [("macro_f1", "Macro-F1"), ("balanced_accuracy", "Balanced Accuracy"), ("log_loss", "Log-Loss")]:
        sub = tuned_results[(tuned_results["split"] == split) & (tuned_results["stage"] == "C_second_semester")].copy()
        ascending = metric == "log_loss"
        sub = sub.sort_values(metric, ascending=ascending)
        plt.figure(figsize=(9, 5.5))
        plt.barh(sub["model"], sub[metric])
        plt.title(f"Tuned {label} by Model at Stage C ({split.title()})")
        plt.xlabel(label)
        plt.tight_layout()
        plt.savefig(TUNE_PLOTS_DIR / f"{split}_stage_c_{metric}_model_ranking_tuned.png", dpi=220)
        plt.close()

len(list(TUNE_PLOTS_DIR.glob("*.png")))

## 10. Report

In [ ]:
report_lines = []
report_lines.append("# Hyperparameter Tuning Report")
report_lines.append("")
report_lines.append("This tuning step selected the best parameter setting for each top model family within each early-warning stage using validation Macro-F1 as the main criterion.")
report_lines.append("")
report_lines.append("## Best Tuned Model By Stage")
for _, row in best_by_stage.iterrows():
    report_lines.append(
        f"- {row['stage']}: {row['model']} with validation Macro-F1={row['macro_f1']:.4f}, "
        f"Balanced Accuracy={row['balanced_accuracy']:.4f}, Log-Loss={row['log_loss']:.4f}."
    )
report_lines.append("")
report_lines.append("## Main Outputs")
report_lines.append("- `csv/tuning_candidate_validation_results.csv`")
report_lines.append("- `csv/tuned_model_comparison.csv`")
report_lines.append("- `csv/best_tuned_validation_model_by_stage.csv`")
report_lines.append("- `csv/tuned_overfitting_gap_table.csv`")
report_lines.append("- `plots/`")

report_path = TUNE_REPORTS_DIR / "hyperparameter_tuning_report.md"
report_path.write_text("\n".join(report_lines), encoding="utf-8")
report_path